# DBT

The Data Build Tool (DBT) is a popular framework for building data manipulation systems.

It's really usefull to check sometimes the [reference to the `dbt` CLI](https://docs.getdbt.com/reference/dbt-commands).

## Compilation

Compilation is a proccess of converting DBT language into scripts specific to the target.

The `dbtf compile` initiates the compilation. It produces the outputs to the `target` folderлл.

---

The following cell inititalises the example project.

In [45]:
#init
dbtf init --project-name knowledge --profile knowledge -q
cd knowledge

Info Created .vscode/extensions.json with dbt extension recommendation
Success Project created successfully!
Info Project name: knowledge
Info Project directory: knowledge
Validating profile inputs, adapters, and connection



The template with jinja templates and compilation.

In [37]:
# file knowledge/models/script.sql
CREATE TABLE 
    {% if target.name == 'dev' %} 
        dev_table
    {% else %} 
        prod_table
    {% endif %}
(
    some_value INTEGER
)

In [38]:
dbtf compile -q

 Rendering [━━━━                ] 9/46 8 succeeded | 37 in-progress             
 Rendering [━━━━                ] 9/46 8 succeeded | 37 in-progress
 Rendering [━━━━                ] 9/46 8 succeeded | 37 in-progress     
 Rendering [━━━━╾               ] 10/46 9 succeeded | 36 in-progress
 Rendering [━━━━━               ] 11/46 10 succeeded | 35 in-progress
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━━━━─           ] 19/46 18 succeeded | 27 in-progress   
 Rendering [━━━━━━━━─           ] 19/46 18 succeeded | 27 in-progress   
 Rendering [━━━━━━━━─           ] 19/46 18 succeeded | 27 in-progress   
 Analyzing [━━━━━━━━━━━━━━━━━╾  ] 35/40 34 succeeded                                


The content of the `target` folder:

In [39]:
ls target

compiled  generic_tests  schemas                 snapshots
data      manifest.json  semantic_manifest.json


The "compiled" sql script which uses table defined in case `target.name == 'dev'` (default case).

In [40]:
cat target/compiled/models/script.sql

CREATE TABLE 
     
        dev_table
    
(
    some_value INTEGER
)


With `-t` parameter you can specify the target to use:

In [42]:
dbtf compile -q -t prod

 Rendering [━━━━━               ] 11/46 10 succeeded | 35 in-progress           
 Rendering [━━━━━               ] 11/46 10 succeeded | 35 in-progress
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━─              ] 12/46 11 succeeded | 34 in-progress   
 Rendering [━━━━━━━━━╾          ] 22/46 21 succeeded | 24 in-progress   
 Analyzing [━━━━━━━╾            ] 15/40 14 succeeded | 8 in-progress                


Now script uses the value defined for `target.name == 'prod'`:

In [44]:
cat target/compiled/models/script.sql

CREATE TABLE 
     
        prod_table
    
(
    some_value INTEGER
)


## Run

To run dbt models use the commands `dbt run`, `dbtf run`. DBT automatically wraps your models in the required DDL (data definition language) and creates the corresponding table or view for each one.

---

Consider the example of running a dbt project. Following cell creates a project and defies the simple model in it.

In [1]:
#init
dbtf init --project-name knowledge --profile knowledge -q
cd knowledge

Info Created .vscode/extensions.json with dbt extension recommendation
Success Project created successfully!
Info Project name: knowledge
Info Project directory: knowledge
Validating profile inputs, adapters, and connection



In [9]:
# file knowledge/models/script.sql
SELECT val1, val2
FROM (
    VALUES 
    (10, 20),
    (30, 40)
) AS temp(val1, val2)

Running:

In [10]:
dbtf run --select script -q

   Running [━━━━━━━━━━━━━━━━━━━━] 1/1                                           


**Note**: `--select` is used to specify which model have to be used.

The following cell shows the database we are using as an example now have corresponding data.

In [11]:
PGPASSWORD=postgres psql -h localhost -U postgres -c "SELECT * FROM public.script"

 val1 | val2 
------+------
   10 |   20
   30 |   40
(2 rows)



## Show

The `dbt show` command allows you to preview the output of a given dbt model.

Check description in the [About dbt show commannd](https://docs.getdbt.com/reference/commands/show?version=2.0).

---

The following cells create a sample project and some model in it.

In [1]:
#init
dbtf init --project-name knowledge --profile knowledge -q
cd knowledge

Info Created .vscode/extensions.json with dbt extension recommendation
Success Project created successfully!
Info Project name: knowledge
Info Project directory: knowledge
Validating profile inputs, adapters, and connection



In [2]:
# file knowledge/models/script.sql
SELECT 'some data' AS col

The output of the `dbt show` for the model:

In [5]:
dbt show -q --select script.sql

| col       |
| --------- |
| some data |



Fusion supports the `show` command as well:

In [4]:
dbtf show -q --select script.sql

Query show_model_knowledge_script] 1/1                                           
┌───────────┐
│ col       │
╞═══════════╡
│ some data │
└───────────┘
1 rows.
m   Running [━━━━━━━━━━━━━━━━━━━━] 1/1                                           


## Models

In dbt, models are SQL scripts with Jinja templates. During execution of the project, they **materialize** (produce) the essences (tables, views, materizlized views, etc.) in the target database systems.

---

The following cell initialise the project.

In [1]:
#init
dbtf init --project-name knowledge --profile knowledge -q
cd knowledge

Info Created .vscode/extensions.json with dbt extension recommendation
Success Project created successfully!
Info Project name: knowledge
Info Project directory: knowledge
Validating profile inputs, adapters, and connection



The definition of the model:

In [10]:
# file knowledge/models/script.sql
SELECT 'some data' AS col, 'new data' AS col2

This "model" simply selects hard-coded values from the database.

In [11]:
dbtf run -q --select script

   Running [                    ] 0/1 1 in-progress                             
   Running [━━━━━━━━━━━━━━━━━━━━] 1/1             
   model:script [0.0s]                                                          


The table with same name then appears in the target PostgreSQL database.

In [12]:
PGPASSWORD=postgres psql -h localhost -U postgres -c "select * from public.script"

    col    |   col2   
-----------+----------
 some data | new data
(1 row)

